# Parallel inversions
**18 September 2026 by MiniUFO**

---
[TOC]

---

## 1. Introduction
Inversion based on successive-over-relaxation (SOR) for a very large-scale problem can be time-consuming.  There are two cases that may leverage the powers of **multicore-CPU** and **advanced CUDA GPU**, in the example of 2D horizontal Poisson equation:

- **Coarse-grained parallelism:** If we have a multi-timestep dataset of vorticity, and we need to invert the streamfunction every timestep, then we could make use of **multicore CPU**.  Each core will handle the inversion of a single timestep (2D horizontal slice), and thus we could speedup the whole calculations by using more cores.  But for a single timestep, there is no speedup at all;
- **Fine-grained parallelism:** For a single timestep, we could make use of the [red-black SOR](https://link.springer.com/chapter/10.1007/978-3-642-31464-3_60) algorithm to further accelerate on **advanced CUDA GPU**.  This is much faster than CPU's serial calculation when the size of the problem is large.

These two ways could be combined together, but we need more tests to verify that.  Here we show these two cases first.

---
## 2. Coarse-grained parallelism
We make use of `SODA_curl.nc` dataset on this repo, which contains 12 timestep.  Normally, to invert streamfunction for these 12 timesteps, we need to do inversion 12 times using a single core of CPU.

In [1]:
import numpy as np
import xarray as xr
import dask
import time
from dask.distributed import Client
import os
import sys
sys.path.append('../../../')

ds = xr.open_dataset('../../../Data/SODA_curl.nc')
print(ds)

<xarray.Dataset> Size: 13MB
Dimensions:  (time: 12, lat: 300, lon: 720)
Coordinates:
  * time     (time) datetime64[ns] 96B 2000-01-01 2000-02-01 ... 2000-12-01
  * lat      (lat) float32 1kB -75.25 -74.75 -74.25 -73.75 ... 73.25 73.75 74.25
  * lon      (lon) float32 3kB 0.25 0.75 1.25 1.75 ... 358.2 358.8 359.2 359.8
    dxF      (lat, lon) float32 864kB ...
    dyF      (lat, lon) float32 864kB ...
    rA       (lat, lon) float32 864kB ...
Data variables:
    curl     (time, lat, lon) float32 10MB ...


### 2.1 Baseline test
First we invert the 12-timestep 2D lat/lon Poisson equation for streamfunction using a single CPU.  This is essentially a single loop over time dimension.  To make this happen, one **CANNOT** chunk the dataset using `chunks={'time':1}`.  So from the information printed above, the dataset does not have a dask backend.

In [2]:
from xinvert import invert_Poisson

def solve(curl, mxLoop=20000, printInfo=False, architect='cpu'):
    """Solve all time steps serially (no dask, single thread)."""
    iParams = {
        'BCs'      : ['extend', 'periodic'],
        'undef'    : np.nan,
        'mxLoop'   : mxLoop,
        'tolerance': 1e-15,
        'printInfo': printInfo,
        'debug'    : False,
        'architect': architect,
    }
    sf = invert_Poisson(curl, dims=['lat', 'lon'], coords='lat-lon', iParams=iParams)
    
    return sf.compute()

In [3]:
# start testing
print('=' * 72)
ntime, nlat, nlon = ds.curl.shape
print(f'Demo No. 1 :  Baseline serial calculation (single-core CPU)')
print(f'  Problem  :  Poisson equation  (wind-stress curl -> streamfunction)')
print(f'  Dataset  :  SODA_curl.nc')
print(f'  grid-size:  {nlat} x {nlon}  ({nlat * nlon:,} points per timestep)')
print(f'  timesteps:  {ntime}')
print(f'  core used:  1')
print('=' * 72)

t0 = time.perf_counter()
sf_ser = solve(ds.curl, printInfo=True)
t_serial = time.perf_counter() - t0
print(f'\nBaesline (serial) time:  {t_serial:.2f} s, {t_serial / 12:.2f} s each timestep')

Demo No. 1 :  Baseline serial calculation (single-core CPU)
  Problem  :  Poisson equation  (wind-stress curl -> streamfunction)
  Dataset  :  SODA_curl.nc
  grid-size:  300 x 720  (216,000 points per timestep)
  timesteps:  12
  CPU used :  1
{time: 2000-01-01T00:00:00} loops 20000, tolerance is 6.511211e-09
{time: 2000-02-01T00:00:00} loops 20000, tolerance is 3.174071e-09
{time: 2000-03-01T00:00:00} loops 20000, tolerance is 1.104921e-09
{time: 2000-04-01T00:00:00} loops 20000, tolerance is 1.129693e-08
{time: 2000-05-01T00:00:00} loops 20000, tolerance is 1.242035e-08
{time: 2000-06-01T00:00:00} loops 20000, tolerance is 9.717440e-09
{time: 2000-07-01T00:00:00} loops 20000, tolerance is 1.079245e-08
{time: 2000-08-01T00:00:00} loops 20000, tolerance is 1.003555e-09
{time: 2000-09-01T00:00:00} loops 20000, tolerance is 1.962126e-09
{time: 2000-10-01T00:00:00} loops 20000, tolerance is 1.937243e-10
{time: 2000-11-01T00:00:00} loops 20000, tolerance is 5.688453e-09
{time: 2000-12-01T0

From the results, one can see that the inversion for each timestep requires **30 seconds**.  Inverting the whole dataset (12 timesteps) requires **6 minutes**.  We need to make use of **multicore CPUs** through dask to accelerate the whole calculations.

### 2.2 Coarse-grained parallelism test
Now we use more CPU cores to parallel inversion among different timesteps.  This requires **dask backend** for the dataset.

In [3]:
ds_dask = xr.open_dataset('../../../Data/SODA_curl.nc', chunks={'time': 1})

In [4]:
from xinvert import invert_Poisson

# start testing
print('=' * 72)
ntime, nlat, nlon = ds.curl.shape
print(f'Demo No. 2 :  Coarse-grained parallelism (multi-core CPU)')
print(f'  Problem  :  Poisson equation  (wind-stress curl -> streamfunction)')
print(f'  Dataset  :  SODA_curl.nc')
print(f'  grid-size:  {nlat} x {nlon}  ({nlat * nlon:,} points per timestep)')
print(f'  timesteps:  {ntime}')
print(f'  core used:  12')
print('=' * 72)

t0 = time.perf_counter()
sf_dsk = solve(ds_dask.curl, printInfo=True) # The only difference is using dask-backend dataset !!!!
t_serial = time.perf_counter() - t0
print(f'\nCoarse-grained parallelism (multi-core CPU) time:  {t_serial:.2f} s, {t_serial / 12:.2f} s each timestep')

Demo No. 2 :  Coarse-grained parallelism (multi-core CPU)
  Problem  :  Poisson equation  (wind-stress curl -> streamfunction)
  Dataset  :  SODA_curl.nc
  grid-size:  300 x 720  (216,000 points per timestep)
  timesteps:  12
  core used:  12
{time: 2000-11-01T00:00:00} loops 20000, tolerance is 5.688453e-09
{time: 2000-06-01T00:00:00} loops 20000, tolerance is 9.717440e-09
{time: 2000-01-01T00:00:00} loops 20000, tolerance is 6.511211e-09
{time: 2000-03-01T00:00:00} loops 20000, tolerance is 1.104921e-09
{time: 2000-08-01T00:00:00} loops 20000, tolerance is 1.003555e-09
{time: 2000-05-01T00:00:00} loops 20000, tolerance is 1.242035e-08
{time: 2000-09-01T00:00:00} loops 20000, tolerance is 1.962126e-09
{time: 2000-10-01T00:00:00} loops 20000, tolerance is 1.937243e-10
{time: 2000-07-01T00:00:00} loops 20000, tolerance is 1.079245e-08
{time: 2000-02-01T00:00:00} loops 20000, tolerance is 3.174071e-09
{time: 2000-12-01T00:00:00} loops 20000, tolerance is 1.171691e-09
{time: 2000-04-01T00

This is a parallel inversion using 12 CPU cores.  The print for each timestep is random, and 12 inversions are done in **~30 seconds**, like a single timestep.  Note that this will use **all available cores** and can be memory consuming.  To control the cores to be used, one need dask client.

In [5]:
from xinvert import invert_Poisson

# start testing
print('=' * 72)
ntime, nlat, nlon = ds.curl.shape
print(f'Demo No. 3 :  Coarse-grained parallelism (multi-core CPU)')
print(f'  Problem  :  Poisson equation  (wind-stress curl -> streamfunction)')
print(f'  Dataset  :  SODA_curl.nc')
print(f'  grid-size:  {nlat} x {nlon}  ({nlat * nlon:,} points per timestep)')
print(f'  timesteps:  {ntime}')
print('=' * 72)
print('')

def run(threads=1, printInfo=False):
    client = Client(n_workers=1, threads_per_worker=threads, dashboard_address=":0") # 1 worker and 4 threads
    
    t0 = time.perf_counter()
    sf = solve(ds_dask.curl, printInfo=printInfo) # using dask-backend dataset !!!!
    t_serial = time.perf_counter() - t0
    print(f'Coarse-grained parallelism ({threads}-core CPU) time:  {t_serial:.2f} s, {t_serial / 12:.2f} s each timestep')
    
    client.close()

    return sf

sf_dsk = run(1)
sf_dsk = run(2)
sf_dsk = run(3)
sf_dsk = run(4)
sf_dsk = run(6)
sf_dsk = run(12)

Demo No. 3 :  Coarse-grained parallelism (multi-core CPU)
  Problem  :  Poisson equation  (wind-stress curl -> streamfunction)
  Dataset  :  SODA_curl.nc
  grid-size:  300 x 720  (216,000 points per timestep)
  timesteps:  12

Coarse-grained parallelism (1-core CPU) time:  369.76 s, 30.81 s each timestep
Coarse-grained parallelism (2-core CPU) time:  189.33 s, 15.78 s each timestep
Coarse-grained parallelism (3-core CPU) time:  128.79 s, 10.73 s each timestep
Coarse-grained parallelism (4-core CPU) time:  97.64 s, 8.14 s each timestep
Coarse-grained parallelism (6-core CPU) time:  66.80 s, 5.57 s each timestep
Coarse-grained parallelism (12-core CPU) time:  36.07 s, 3.01 s each timestep


### 2.3 Fine-grained parallelism test
Now we use CUDA GPU to accelerate inversion for a single timestep.  This requires **a CUDA GPU** on your machine.

In [8]:
def _gpu_available():
    try:
        from numba import cuda
        if cuda.is_available():
            return True
        # real probe
        @cuda.jit
        def _t(x):
            x[0] = 1.0
        a = cuda.device_array(1, dtype=np.float64)
        _t[1, 1](a)
        return a.copy_to_host()[0] == 1.0
    except Exception:
        return False


def _gpu_name():
    try:
        from numba import cuda
        d = cuda.get_current_device()
        name = d.name
        if isinstance(name, bytes):
            name = name.decode()
        return name
    except Exception:
        return 'unknown'

In [9]:
gpu_ok = _gpu_available()

print('=' * 72)
print('Demo 2: Large-scale Poisson solve on GPU')
print('  Problem: Poisson equation  (wind-stress curl -> streamfunction)')
print('  Data:    SODA_curl.nc  (real wind-stress curl, upsampled)')
print('=' * 72)
print(f'  GPU: {"available" if gpu_ok else "NOT available (CPU-only)"}')
if gpu_ok:
    print(f'  GPU model: {_gpu_name()}')
print(f'  CPU cores: {os.cpu_count()}')
print()

ntimes, nlat0, nlon0 = ds.curl.shape
print(f'  base grid: {nlat0} x {nlon0}  ({nlat0 * nlon0:,} points)')

Demo 2: Large-scale Poisson solve on GPU
  Problem: Poisson equation  (wind-stress curl -> streamfunction)
  Data:    SODA_curl.nc  (real wind-stress curl, upsampled)
  GPU: available
  GPU model: NVIDIA GeForce RTX 3090
  CPU cores: 112

  base grid: 300 x 720  (216,000 points)


In [10]:
sf_gpu = solve(ds.curl[0], architect='gpu')

/home/qianyk/miniconda3/envs/py13/lib/python3.13/site-packages/numba_cuda/numba/cuda/dispatcher.py:748: NumbaPerformanceWarning: Grid size 3 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/home/qianyk/miniconda3/envs/py13/lib/python3.13/site-packages/numba_cuda/numba/cuda/dispatcher.py:748: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
